# Stage 2 — Data Cleaning + Feature Identification
**Capstone: Multimodal Ocular Disease Prediction (Myopia focus — Experiment 1)**

Pipeline position: `full_df.csv` → Data Exploration → **Data Cleaning + Feature Identification** → Normalization → Feature Selection → Patient-level Split → Classification → Evaluation

This notebook does NOT train any model. It:
1. Inspects every one of the 19 columns and classifies each as: **feature**, **target**, **key**, or **excluded (leakage / not usable)**.
2. Investigates and resolves the `Patient Age == 1` data-quality issue found in Stage 1.
3. Investigates a new issue found while doing this: the `labels`/`target` column does not always agree with the one-hot `N,D,G,C,A,H,M,O` columns.
4. Builds a clean modelling dataset for Experiment 1 (Myopia vs Non-Myopia) and saves it to a **new file**, leaving `full_df.csv` untouched.

Everything below is executed against the real dataset — no numbers are assumed.

## 1. Load data (unchanged from Stage 1)

Project folder layout:
```
Capstone Datasets/
├── Data/            <- full_df.csv lives here; myopia_baseline_clean.csv is saved here too
├── Code/
│   └── notebooks/   <- this notebook lives here
└── Documentation/
```

In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

CANDIDATE_PATHS = [
    "../../Data/full_df.csv",
    "Data/full_df.csv",
    "full_df.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("full_df.csv not found. Checked: " + ", ".join(CANDIDATE_PATHS))

# Where to save the cleaned output — same folder full_df.csv was found in
DATA_DIR = os.path.dirname(DATA_PATH) or "."
OUT_PATH = os.path.join(DATA_DIR, "myopia_baseline_clean.csv")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "— shape:", df.shape)
assert df.shape == (6392, 19), "Shape does not match Stage 1 — stop and check your file."


Loaded: ../../Data/full_df.csv — shape: (6392, 19)


## 2. Column-by-column classification

Every one of the 19 original columns, decided by what it actually is (not by name alone):

| # | Column | Role | Reason |
|---|--------|------|--------|
| 1 | `ID` | Key | Patient identifier — needed later for patient-grouped train/test split. Not a predictive feature. |
| 2 | `Patient Age` | **Feature** | Legitimate pre-diagnosis demographic variable (after cleaning — see Section 3). |
| 3 | `Patient Sex` | **Feature** | Legitimate pre-diagnosis demographic variable. |
| 4 | `Left-Fundus` | Excluded | Image filename, not a clinical measurement. Relevant only in the later image stage. |
| 5 | `Right-Fundus` | Excluded | Same as above. |
| 6 | `Left-Diagnostic Keywords` | Excluded — leakage | Free-text diagnosis, effectively states the answer. |
| 7 | `Right-Diagnostic Keywords` | Excluded — leakage | Same as above. |
| 8 | `N` | Excluded — leakage (for Exp. 1) | Diagnosis-derived flag for a *different* condition, produced by the same clinical read that produced the myopia label. Kept in the cleaned file as reference for the future multi-class experiment, not used as an Experiment-1 input. |
| 9 | `D` | Excluded — leakage (for Exp. 1) | Same reasoning as `N`. |
| 10 | `G` | Excluded — leakage (for Exp. 1) | Same reasoning as `N`. |
| 11 | `C` | Excluded — leakage (for Exp. 1) | Same reasoning as `N`. |
| 12 | `A` | Excluded — leakage (for Exp. 1) | Same reasoning as `N`. |
| 13 | `H` | Excluded — leakage (for Exp. 1) | Same reasoning as `N`. |
| 14 | `M` | **Target (reference)** | The one-hot myopia flag. Kept, but see Section 4 — it disagrees with the official label in 74 rows, so it is not used directly as the final target. |
| 15 | `O` | Excluded — leakage (for Exp. 1) | Same reasoning as `N`. |
| 16 | `filepath` | Excluded | File path string, not a feature. |
| 17 | `labels` | Used to derive target, then excluded | Official per-patient diagnosis label. Used in Section 4 to build the final, trustworthy myopia target — not used as a model input itself. |
| 18 | `target` | Excluded | One-hot-encoded restatement of `labels`. Same information, not an input feature. |
| 19 | `filename` | Excluded | Image filename, not a clinical measurement. |

**Bottom line for Experiment 1 (Myopia vs Non-Myopia):** after removing every column that is an identifier, an image reference, free-text diagnosis, or a diagnosis-derived flag, the dataset provides exactly **two** legitimate predictive features: `Patient Age` and `Patient Sex`. There are no other clinical measurements (no refractive error, no intraocular pressure, no family history, etc.) in this CSV. This is a real limitation of the dataset, not a mistake in the cleaning — it's recorded in the documentation as a limitation to state honestly in the report.

In [2]:
feature_cols = ["Patient Age", "Patient Sex"]
key_cols = ["ID"]
reference_disease_cols = ["N", "D", "G", "C", "A", "H", "O"]  # kept for future multi-class stage, not used in Exp.1
excluded_cols = ["Left-Fundus", "Right-Fundus", "Left-Diagnostic Keywords",
                  "Right-Diagnostic Keywords", "filepath", "target", "filename"]

print("Experiment 1 model input features:", feature_cols)
print("Key column (not a feature):", key_cols)
print("Reference-only columns (kept, not used in Exp.1):", reference_disease_cols)
print("Fully excluded columns:", excluded_cols)


Experiment 1 model input features: ['Patient Age', 'Patient Sex']
Key column (not a feature): ['ID']
Reference-only columns (kept, not used in Exp.1): ['N', 'D', 'G', 'C', 'A', 'H', 'O']
Fully excluded columns: ['Left-Fundus', 'Right-Fundus', 'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'filepath', 'target', 'filename']


## 3. Investigate `Patient Age == 1`

Stage 1 flagged 28 rows with `Patient Age == 1`. Before deciding how to treat them, look at who they are.

In [3]:
age1 = df[df["Patient Age"] == 1]
print("Rows with age == 1:", len(age1))
print("Unique patients affected:", age1["ID"].nunique())
print()
print("Minimum age in the rest of the dataset (age != 1):", df[df["Patient Age"] != 1]["Patient Age"].min())
print("--> There is a gap: nobody in this dataset is aged 2-13. That's a strong signal age=1 is a placeholder, not a real age.")


Rows with age == 1: 28
Unique patients affected: 15

Minimum age in the rest of the dataset (age != 1): 14
--> There is a gap: nobody in this dataset is aged 2-13. That's a strong signal age=1 is a placeholder, not a real age.


In [4]:
disease_cols = ["N", "D", "G", "C", "A", "H", "M", "O"]
print("Disease flags among the age==1 rows:")
print(age1[disease_cols].sum())
print()
pct_myopia = (age1["M"] == 1).mean() * 100
print(f"{pct_myopia:.0f}% of age==1 rows are flagged Myopia (M=1), vs "
      f"{(df['M']==1).mean()*100:.1f}% in the dataset overall.")


Disease flags among the age==1 rows:
N     0
D     0
G     0
C     0
A     2
H     0
M    25
O     2
dtype: int64

89% of age==1 rows are flagged Myopia (M=1), vs 4.8% in the dataset overall.


**Finding:** age=1 is not spread evenly across the dataset — it is heavily concentrated in myopia-flagged rows (25 of 28 rows, ~89%, vs ~4.8% myopia prevalence overall). Combined with the fact that no real age between 2 and 13 exists anywhere else in the data, this is almost certainly a placeholder for "age not recorded," used disproportionately for a specific sub-group of myopia patients — not a genuine age value.

**Decision:** treat `Patient Age == 1` as **missing**, not as a real age.

- Set these 28 values to `NaN` in the cleaned dataset.
- Actual imputation (e.g. filling with the training-set median) is deferred to the modelling stage, and must be fit on the training split only, to avoid leaking information across the patient-level train/test split.
- We are **not** adding an "age was missing" indicator as a model feature. Normally that's good practice, but here it would almost perfectly encode the myopia label itself (since 89% of missing-age rows are myopia-positive) — that's a shortcut created by how this specific dataset was collected, not a real clinical signal, so using it would artificially inflate Experiment 1 accuracy. It's recorded in the cleaned file for transparency (`Age_Was_Missing` column) but is excluded from the model input feature list.

In [5]:
df["Patient Age Cleaned"] = df["Patient Age"].replace(1, np.nan)
df["Age_Was_Missing"] = (df["Patient Age"] == 1).astype(int)

print("Missing values in 'Patient Age Cleaned' now:", df["Patient Age Cleaned"].isna().sum())
print("Age range after cleaning (ignoring NaN):", df["Patient Age Cleaned"].min(), "-", df["Patient Age Cleaned"].max())


Missing values in 'Patient Age Cleaned' now: 28
Age range after cleaning (ignoring NaN): 14.0 - 91.0


## 4. A second issue found during cleaning: `labels` vs the one-hot columns disagree

While preparing the target column, the official `labels` field (the ODIR per-patient diagnosis list) was compared against the one-hot `N,D,G,C,A,H,M,O` columns. They should describe the same thing. They don't always.

In [6]:
import ast

def parse_labels(s):
    try:
        return set(ast.literal_eval(s))
    except Exception:
        return set()

df["_official_labels"] = df["labels"].apply(parse_labels)
df["_onehot_set"] = df[disease_cols].apply(lambda r: set(c for c in disease_cols if r[c] == 1), axis=1)

mismatch_flags = [a != b for a, b in zip(df["_official_labels"], df["_onehot_set"])]
n_mismatch = sum(mismatch_flags)
print(f"Rows where official labels and one-hot columns disagree: {n_mismatch} / {len(df)} "
      f"({n_mismatch/len(df)*100:.1f}%)")


Rows where official labels and one-hot columns disagree: 1725 / 6392 (27.0%)


In [7]:
df["_official_M"] = df["_official_labels"].apply(lambda s: "M" in s)

onehot_m_count = (df["M"] == 1).sum()
official_m_count = df["_official_M"].sum()
only_onehot = ((df["M"] == 1) & (~df["_official_M"])).sum()
only_official = ((df["M"] == 0) & (df["_official_M"])).sum()

print("Myopia count, one-hot M column:      ", onehot_m_count)
print("Myopia count, official labels column:", official_m_count)
print("Rows: onehot says Myopia, official does NOT:", only_onehot)
print("Rows: official says Myopia, onehot does NOT:", only_official)


Myopia count, one-hot M column:       306
Myopia count, official labels column: 232
Rows: onehot says Myopia, official does NOT: 74
Rows: official says Myopia, onehot does NOT: 0


**Finding:** the one-hot `M` column flags 306 rows as myopia; the official `labels` column flags 232. All 74 of the extra one-hot rows are cases where a keyword like "pathological myopia" appears in one eye's notes but the official per-patient label did not count it (e.g. because the other eye was normal, or the official ODIR labelling rule didn't count that particular keyword). There are zero rows where the official label says myopia but the one-hot column disagrees — the one-hot version is a superset.

**232 also matches the myopia count already recorded in the original project brief**, which is independent evidence that the official `labels` column is the correct ground truth to model against, and the one-hot `M` column is a looser, independently keyword-matched version.

**Decision:** the final myopia target for Experiment 1 is derived from the **official `labels` column**, not the one-hot `M` column. The one-hot `M` is kept in the cleaned file as `Myopia_OneHot_Reference` for transparency, but it is not the modelling target.

In [8]:
df["Myopia"] = df["_official_M"].astype(int)
print(df["Myopia"].value_counts())
print((df["Myopia"].value_counts(normalize=True) * 100).round(2).astype(str) + "%")


Myopia
0    6160
1     232
Name: count, dtype: int64
Myopia
0    96.37%
1     3.63%
Name: proportion, dtype: str


## 5. Other data-quality checks

In [9]:
# Drop the temporary set-valued helper columns first — they can't be hashed for duplicate checks
df = df.drop(columns=["_official_labels", "_onehot_set"])

# Duplicate rows (repeat from Stage 1, confirming still true)
print("Fully duplicated rows:", df.duplicated().sum())

# filename consistency
match = (df["filename"] == df["Left-Fundus"]) | (df["filename"] == df["Right-Fundus"])
print("filename matches Left- or Right-Fundus:", match.sum(), "/", len(df))

# Patient Sex integrity
print("Patient Sex values:", df["Patient Sex"].unique().tolist())
print("Patient Sex missing:", df["Patient Sex"].isna().sum())

# Age sanity after cleaning
print()
print("Patient Age Cleaned — describe:")
print(df["Patient Age Cleaned"].describe())


Fully duplicated rows: 0
filename matches Left- or Right-Fundus: 6392 / 6392
Patient Sex values: ['Female', 'Male']
Patient Sex missing: 0

Patient Age Cleaned — describe:
count    6364.000000
mean       58.108108
std        11.129089
min        14.000000
25%        51.000000
50%        59.000000
75%        66.000000
max        91.000000
Name: Patient Age Cleaned, dtype: float64


No new issues found beyond what's already documented: 0 duplicate rows, filenames are internally consistent, `Patient Sex` has exactly two clean categories with no missing values, and the cleaned age range (14–91) is sane.

## 6. Build the clean modelling dataset

Saved as a **new file**, `myopia_baseline_clean.csv` — `full_df.csv` is not modified.

Columns kept:
- `ID` — key, for patient-grouped splitting later (not a feature)
- `Patient Age Cleaned`, `Patient Sex` — the two Experiment-1 model input features
- `Age_Was_Missing` — transparency flag, excluded from the model input list (see Section 3)
- `Myopia` — final target for Experiment 1 (from official labels)
- `Myopia_OneHot_Reference` — original `M` column, kept for reference only
- `N, D, G, C, A, H, O`, `n_conditions` — reference columns for the future multi-class experiment (Experiment 2), not used as inputs here

In [10]:
df["n_conditions"] = df[disease_cols].sum(axis=1)

clean_df = df[[
    "ID",
    "Patient Age Cleaned",
    "Patient Sex",
    "Age_Was_Missing",
    "Myopia",
    "M",
    "N", "D", "G", "C", "A", "H", "O",
    "n_conditions",
]].rename(columns={
    "Patient Age Cleaned": "Patient_Age",
    "M": "Myopia_OneHot_Reference",
})

print("Clean dataset shape:", clean_df.shape)
clean_df.head()


Clean dataset shape: (6392, 14)


,ID,Patient_Age,Patient Sex,Age_Was_Missing,Myopia,Myopia_OneHot_Reference,N,D,G,C,A,H,O,n_conditions
0,0,69.0,Female,0,0,0,0,0,0,1,0,0,0,1
1,1,57.0,Male,0,0,0,1,0,0,0,0,0,0,1
2,2,42.0,Male,0,0,0,0,1,0,0,0,0,1,2
3,4,53.0,Male,0,0,0,0,1,0,0,0,0,1,2
4,5,50.0,Female,0,0,0,0,1,0,0,0,0,0,1


In [11]:
clean_df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

# Verify: re-read it back and confirm it matches what we just built
check = pd.read_csv(OUT_PATH)
print("Re-read shape:", check.shape)
assert check.shape == clean_df.shape
assert list(check.columns) == list(clean_df.columns)
print("Verified: file on disk matches the in-memory cleaned dataset.")

# Confirm original file is untouched
orig_check = pd.read_csv(DATA_PATH)
assert orig_check.shape == (6392, 19)
print("Verified: full_df.csv is unchanged —", orig_check.shape)


Saved: ../../Data/myopia_baseline_clean.csv
Re-read shape: (6392, 14)
Verified: file on disk matches the in-memory cleaned dataset.
Verified: full_df.csv is unchanged — (6392, 19)


## 7. Final feature list for Experiment 1 (Myopia vs Non-Myopia)

| Role | Column(s) |
|------|-----------|
| Model input features | `Patient_Age` (missing values still present — to be imputed on the training split only, in the next stage), `Patient Sex` (to be encoded in the next stage) |
| Target | `Myopia` (0 = Non-Myopia, 1 = Myopia) |
| Key (not a feature) | `ID` (for patient-grouped train/test split) |
| Excluded from input | Everything diagnosis-derived, text, or file-path related (see Section 2 table); `Age_Was_Missing` is excluded from the model for the reason in Section 3 |
| Kept for later (Experiment 2 only) | `N, D, G, C, A, H, O`, `n_conditions`, `Myopia_OneHot_Reference` |

**Remaining concerns to carry into the next stage:**
1. `Patient_Age` still has 28 missing values (by design) — must be imputed using the training split only.
2. `Patient Sex` is still a raw string category — needs encoding in the preprocessing stage.
3. The feature set is very thin (only age and sex) — this is a genuine dataset limitation, not a cleaning oversight, and should be stated plainly in the report rather than glossed over.
4. Experiment 1 is imbalanced (~4–5% positive class) — confirmed again here with the corrected target; class weighting or an equivalent technique will be needed at modelling time.
